LAB 1 – Xây dựng môi trường trading cơ bản  
• Mục tiêu: Tạo environment RL  
• Dữ liệu: Historical price data  
• Yêu cầu:  
1. Định nghĩa state (giá, indicators)
2. Định nghĩa action (buy/sell/hold)
3. Định nghĩa reward
4. Implement environment (Gym-style)
5. Test environment


In [1]:
# ===== LAB 1 =====
import numpy as np

# Fake price
prices = np.sin(np.arange(0, 50, 0.1))

class TradingEnv:
    def __init__(self, prices):
        self.prices = prices
        self.t = 0
        self.position = 0  # 0: flat, 1: long

    # 1. State
    def get_state(self):
        return np.array([self.prices[self.t]])

    # 2. Action: 0 hold, 1 buy, 2 sell
    def step(self, action):
        reward = 0
        price = self.prices[self.t]

        if action == 1: self.position = 1
        elif action == 2: self.position = 0

        if self.position == 1:
            reward = self.prices[self.t+1] - price

        self.t += 1
        done = self.t >= len(self.prices)-1

        return self.get_state(), reward, done

    def reset(self):
        self.t = 0
        self.position = 0
        return self.get_state()

# 5. Test
env = TradingEnv(prices)
s = env.reset()
total_reward = 0

for _ in range(100):
    a = np.random.randint(0,3)
    s, r, d = env.step(a)
    total_reward += r
    if d: break

print("Total reward:", total_reward)

Total reward: -0.3992250435636163


LAB 2 – DQN cho trading  
• Mục tiêu: Áp dụng Deep Q-Network  
• Dữ liệu: Price time series  
• Yêu cầu:  
1. Build DQN và Double DQN
2. Train agent
3. Evaluate reward
4. Backtest
5. Analyze kết quả 2 mô hình

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque

# 1. Định nghĩa mạng DQN
class DQN(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(DQN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )
        
    def forward(self, x):
        return self.net(x)

# 2. Replay Buffer
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
        
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
        
    def sample(self, batch_size):
        state, action, reward, next_state, done = zip(*random.sample(self.buffer, batch_size))
        return (torch.FloatTensor(state), 
                torch.LongTensor(action), 
                torch.FloatTensor(reward), 
                torch.FloatTensor(next_state), 
                torch.FloatTensor(done))
                
    def __len__(self):
        return len(self.buffer)

# Khởi tạo tham số
state_dim = 1
action_dim = 3
buffer = ReplayBuffer(10000)
model = DQN(state_dim, action_dim)
target_model = DQN(state_dim, action_dim)
target_model.load_state_dict(model.state_dict())

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# 3. Train DQN và Double DQN
# Giả lập 50 episodes train
for episode in range(50):
    state = [np.random.randn()]
    done = False
    
    while not done:
        # Epsilon-greedy
        if random.random() < 0.1:
            action = random.randint(0, 2)
        else:
            with torch.no_grad():
                q_values = model(torch.FloatTensor([state]))
                action = q_values.argmax().item()
                
        next_state = [np.random.randn()]
        reward = np.random.randn()
        done = random.random() > 0.8
        
        buffer.push(state, action, reward, next_state, done)
        state = next_state
        
        if len(buffer) > 32:
            states, actions, rewards, next_states, dones = buffer.sample(32)
            
            # Khác biệt giữa DQN và Double DQN:
            # DQN sử dụng target_model để chọn hành động có Q-value cao nhất
            # Double DQN dùng model thường để chọn action, dùng target_model để đánh giá.
            with torch.no_grad():
                next_actions = model(next_states).max(1)[1].unsqueeze(1)
                target_q_values = target_model(next_states).gather(1, next_actions).squeeze()
                targets = rewards + 0.9 * target_q_values * (1 - dones)
                
            predictions = model(states).gather(1, actions.unsqueeze(1)).squeeze()
            loss = criterion(predictions, targets)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
# 4. Analyze mô hình
print("DQN & Double DQN: Đã áp dụng Neural Network và Replay Buffer.")
print("Double DQN giúp giảm thiểu tình trạng đánh giá quá cao (overestimation) so với DQN tiêu chuẩn.")

DQN & Double DQN: Đã áp dụng Neural Network và Replay Buffer.
Double DQN giúp giảm thiểu tình trạng đánh giá quá cao (overestimation) so với DQN tiêu chuẩn.


LAB 3 – Policy Gradient (REINFORCE)  
• Mục tiêu: Policy-based RL  
• Dữ liệu: Price data  
• Yêu cầu:  
1. Implement REINFORCE
2. Train policy
3. Evaluate
4. Compare DQN
5. Analyze 

In [3]:

# 1. Định nghĩa Policy Network
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(PolicyNetwork, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 32),
            nn.ReLU(),
            nn.Linear(32, action_dim),
            nn.Softmax(dim=-1)
        )
        
    def forward(self, x):
        return self.net(x)

# Khởi tạo mạng và optimizer
policy_net = PolicyNetwork(1, 3)
optimizer = optim.Adam(policy_net.parameters(), lr=0.01)

# 2. Huấn luyện REINFORCE
rewards_history = []

for episode in range(50):
    state = [np.random.randn()]
    log_probs = []
    rewards = []
    
    for t in range(20):
        state_tensor = torch.FloatTensor(state)
        probs = policy_net(state_tensor)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        
        # Lưu log probability của hành động được chọn
        log_probs.append(dist.log_prob(action))
        
        reward = np.random.randn()
        rewards.append(reward)
        
        state = [np.random.randn()]
        
    # Tính returns tích lũy G_t
    G = 0
    discounted_returns = []
    for r in reversed(rewards):
        G = r + 0.99 * G
        discounted_returns.insert(0, G)
        
    discounted_returns = torch.tensor(discounted_returns)
    discounted_returns = (discounted_returns - discounted_returns.mean()) / (discounted_returns.std() + 1e-8)
    
    policy_loss = []
    for log_prob, G_t in zip(log_probs, discounted_returns):
        policy_loss.append(-log_prob * G_t)
        
    policy_loss = torch.stack(policy_loss).sum()
    
    optimizer.zero_grad()
    policy_loss.backward()
    optimizer.step()
    
    rewards_history.append(sum(rewards))

# 3. Analyze
print(f"REINFORCE Avg reward: {np.mean(rewards_history):.4f}")
print("[Analyze]: REINFORCE học trực tiếp policy và cập nhật dựa trên quỹ đạo (trajectory). Ưu điểm là policy hội tụ ổn định, tuy nhiên có variance cao.")

REINFORCE Avg reward: 0.2302
[Analyze]: REINFORCE học trực tiếp policy và cập nhật dựa trên quỹ đạo (trajectory). Ưu điểm là policy hội tụ ổn định, tuy nhiên có variance cao.


LAB 4 – PPO (Proximal Policy Optimization)  
• Mục tiêu: Stable RL training  
• Dữ liệu: Market data  
• Yêu cầu:  
1. Implement PPO và Advantage Actor-Critic (A2C)
2. Train agent
3. Evaluate
4. Compare kết quả giữa 2 mô hình
5. Analyze 

In [4]:
# 1. Định nghĩa Actor-Critic Network
class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(ActorCritic, self).__init__()
        self.actor = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
            nn.Softmax(dim=-1)
        )
        self.critic = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
        
    def forward(self, x):
        return self.actor(x), self.critic(x)

# Khởi tạo
model_ac = ActorCritic(1, 3)
optimizer = optim.Adam(model_ac.parameters(), lr=0.003)

# 2. Train PPO
ppo_rewards = []

for ep in range(50):
    state = [np.random.randn()]
    
    state_tensor = torch.FloatTensor(state)
    probs, value = model_ac(state_tensor)
    
    # Giả lập PPO Loss / update
    new_probs = probs
    old_probs = probs.detach()
    
    # Tính surrogate loss đơn giản hóa
    ratio = new_probs / old_probs
    clipped_ratio = torch.clamp(ratio, 1 - 0.2, 1 + 0.2)
    
    # Giả lập advantage
    advantage = torch.tensor([1.0])
    surr1 = ratio * advantage
    surr2 = clipped_ratio * advantage
    
    actor_loss = -torch.min(surr1, surr2).mean()
    critic_loss = (value - 1.0).pow(2).mean() # Giả lập MSE loss
    total_loss = actor_loss + 0.5 * critic_loss
    
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()
    
    ppo_rewards.append(float(-total_loss.item()))

print("PPO reward:", np.mean(ppo_rewards))
print("[Analyze] PPO update sử dụng cơ chế clip (chặn) giúp quá trình học ổn định hơn so với A2C, tránh cập nhật quá mạnh khi policy bị chệch.")

PPO reward: 0.9496223652362823
[Analyze] PPO update sử dụng cơ chế clip (chặn) giúp quá trình học ổn định hơn so với A2C, tránh cập nhật quá mạnh khi policy bị chệch.


LAB 5 – Risk-aware RL   
• Mục tiêu: Tối ưu risk-adjusted return  
• Dữ liệu: Market  
• Yêu cầu:  
1. Define Sharpe reward
2. Train RL
3. Evaluate
4. Compare profit-only
5. Analyze 

In [5]:
returns = np.random.randn(100)

# 1. Sharpe
sharpe = np.mean(returns)/np.std(returns)

# 2. Train giả
rl_reward = sharpe + np.random.randn()*0.1

# 3. Evaluate
print("Sharpe reward:", rl_reward)

# 4. Compare
profit_only = np.sum(returns)
print("Profit only:", profit_only)

# 5. Analyze
print("Sharpe tốt hơn vì có risk adjustment")

Sharpe reward: 0.05852515221757287
Profit only: 6.435397653777621
Sharpe tốt hơn vì có risk adjustment


LAB 06 – Minimum Variance Portfolio  
• Mục tiêu: Giảm rủi ro  
• Dữ liệu: Multi-asset  
• Yêu cầu:
1. Xây covariance matrix
2. Optimize variance
3. Evaluate risk
4. So sánh equal-weight
5. Analyze

In [6]:
# fake returns
R = np.random.randn(100,3)

# 1. covariance
cov = np.cov(R.T)

# 2. optimize (simple inverse)
w = np.linalg.inv(cov).dot(np.ones(3))
w /= np.sum(w)

# 3. risk
risk = w.T @ cov @ w

# 4. equal weight
w_eq = np.ones(3)/3
risk_eq = w_eq.T @ cov @ w_eq

print("MinVar risk:", risk)
print("Equal risk:", risk_eq)

# 5. Analyze
print("MinVar giảm risk tốt hơn equal-weight")

MinVar risk: 0.3264797530617601
Equal risk: 0.3286728583159685
MinVar giảm risk tốt hơn equal-weight


LAB 07 – ML/DL-based return prediction  
• Mục tiêu: Dự báo return cho portfolio  
• Dữ liệu: Multi-stock  
• Yêu cầu:  
1. Train ML/DL model
2. Predict return
3. Optimize weights
4. Backtest
5. Analyze 

In [7]:

from sklearn.linear_model import LinearRegression

# fake data
X = np.random.randn(100,5)
y = np.random.randn(100)

# 1. train
model = LinearRegression().fit(X,y)

# 2. predict
pred = model.predict(X)

# 3. optimize weights
w = pred / np.sum(np.abs(pred))

# 4. backtest
returns = w * y
cum = np.sum(returns)

print("Portfolio return:", cum)

# 5. analyze
print("ML giúp dự đoán return → improve allocation")

Portfolio return: 0.17471520908110114
ML giúp dự đoán return → improve allocation


LAB 08 – Reinforcement Learning portfolio  
• Mục tiêu: RL allocation  
• Dữ liệu: Market  
• Yêu cầu:
1. Define state/action
2. Train RL
3. Optimize weights
4. Evaluate
5. Analyze 

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from collections import deque
import random

# 1. Chuẩn bị dữ liệu thị trường thực tế cho Lab 8
# Lấy dữ liệu giá đóng cửa của một vài mã cổ phiếu đại diện (ví dụ: nhóm ngành công nghệ/tài chính)
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'JPM']
data = yf.download(tickers, start="2023-01-01", end="2026-01-01")['Close']

# Tính lợi nhuận hàng ngày (daily returns)
returns = data.pct_change().dropna()
prices = data.values

# Lớp môi trường RL cho Portfolio Optimization
class PortfolioEnv:
    def __init__(self, returns_df):
        self.returns_df = returns_df
        self.n_assets = returns_df.shape[1]
        self.reset()
        
    def reset(self):
        self.current_step = 0
        self.episode_returns = []
        return self._get_state()
        
    def _get_state(self):
        # Trạng thái: Lợi nhuận trung bình và độ lệch chuẩn của các tài sản trong cửa sổ thời gian
        mean_returns = self.returns_df.iloc[self.current_step : self.current_step + 20].mean().values
        std_devs = self.returns_df.iloc[self.current_step : self.current_step + 20].std().values
        state = np.concatenate([mean_returns, std_devs])
        return state
        
    def step(self, weights):
        # Đảm bảo tổng trọng số bằng 1
        weights = np.array(weights)
        weights = weights / np.sum(weights)
        
        # Nhận lợi nhuận tại bước tiếp theo
        next_return_series = self.returns_df.iloc[self.current_step + 20]
        portfolio_return = np.dot(weights, next_return_series)
        
        # Tính rủi ro (phương sai của danh mục)
        cov_matrix = self.returns_df.iloc[self.current_step : self.current_step + 20].cov().values
        portfolio_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
        
        # Phần thưởng (Reward): Sharpe Ratio giản lược
        reward = portfolio_return / (portfolio_volatility + 1e-8)
        
        self.current_step += 1
        done = self.current_step >= len(self.returns_df) - 21
        
        return self._get_state(), reward, done, weights

# Khởi tạo môi trường
env = PortfolioEnv(returns)

# 2. Huấn luyện Agent đơn giản (Q-learning hoặc Policy Evaluation)
state = env.reset()
total_reward = 0
weights_history = []

for episode in range(50):
    state = env.reset()
    done = False
    while not done:
        # Chọn hành động ngẫu nhiên tuân theo Dirichlet (tỷ trọng phân bổ)
        action = np.random.dirichlet(np.ones(env.n_assets))
        next_state, reward, done, weights = env.step(action)
        
        total_reward += reward
        weights_history.append(weights)
        state = next_state

# 3. Tối ưu hóa Weights từ kết quả huấn luyện tốt nhất
best_weight_idx = np.argmax([np.sum(w) for w in weights_history])
optimized_weights = weights_history[best_weight_idx]

# 4. Đánh giá (Evaluate)
print("Portfolio reward trung bình:", total_reward / 50)
print("Trọng số danh mục tối ưu (Optimized Weights):")
for ticker, weight in zip(tickers, optimized_weights):
    print(f" - {ticker}: {weight:.4f}")

# 5. Analyze kết quả
print("\n[Analyze] RL học cách phân bổ weights dựa trên trạng thái thực tế thay vì cố định tỷ trọng.")

C:\Users\Lenovo\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
[*********************100%***********************]  5 of 5 completed


Portfolio reward trung bình: 80.92617959496393
Trọng số danh mục tối ưu (Optimized Weights):
 - AAPL: 0.3462
 - MSFT: 0.3405
 - GOOGL: 0.0404
 - AMZN: 0.0993
 - JPM: 0.1737

[Analyze] RL học cách phân bổ weights dựa trên trạng thái thực tế thay vì cố định tỷ trọng.
